# Machine Learning Analysis of stock Market Data clustering and time series forecasting approach

# Lets import libraries

In [23]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Set default figure size
plt.rcParams["figure.figsize"] = (12, 6)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [24]:
import yfinance as yf

# Choose 10 stocks
stocks = ["NFLX", "ORCL", "IBM", "QCOM", "ADBE", "INTU", "CSCO", "TXN", "AMD", "CRM"]

for stock in stocks:
    df = yf.download(stock, start="2016-01-01", end="2025-12-31")
    df.reset_index(inplace=True)   # make Date a normal column 
    df.to_csv(f"{stock}.csv", index=False)
    print(f"{stock} saved")

print("Download complete!")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


NFLX saved
ORCL saved
IBM saved
QCOM saved
ADBE saved
INTU saved
CSCO saved
TXN saved


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

AMD saved
CRM saved
Download complete!


 # Load all CSV files

In [28]:
# list of selscted stock tickers
stock = ["NFLX", "ORCL", "IBM", "QCOM", "ADBE", "INTU", "CSCO", "TXN", "AMD", "CRM"]

# Creat an empty list to store all stock data
all_data = []

for stock in stock:
     # Read each CSV file / CSV
    df = pd.read_csv(f"{stock}.csv")
    # Remove the first two incorrect rows 
    df = df.iloc[2:].copy()

    df.rename(columns={"Price": "Date"}, inplace=True)

    #Add stock name as a new column
    df["Stock"] = stock

    #Store cleaned file in the kist
    all_data.append(df)

#combine all stock data into one dataframe
combined_df = pd.concat(all_data, ignore_index=True)

#Preview the raw combined data
print(combined_df.head())
print(combined_df.shape)
print(combined_df.columns)


         Date               Close                High                 Low  \
0  2016-01-05  10.765999794006348  11.057999610900879  10.585000038146973   
1  2016-01-06  11.767999649047852  11.791000366210938  10.496000289916992   
2  2016-01-07  11.456000328063965  12.218000411987305  11.229000091552734   
3  2016-01-08  11.138999938964844  11.772000312805176  11.109999656677246   
4  2016-01-11  11.496999740600586  11.678999900817871  11.119999885559082   

                 Open     Volume Stock  
0  11.045000076293945  176646000  NFLX  
1  10.529000282287598  330457000  NFLX  
2   11.63599967956543  336367000  NFLX  
3  11.633000373840332  180671000  NFLX  
4  11.213000297546387  219204000  NFLX  
(25120, 7)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Stock'], dtype='str')


In [29]:
# Remove extra space from column names 
combined_df.columns = combined_df.columns.str.strip()

# Convert Date column to datetime format
combined_df["Date"] = pd.to_datetime(combined_df["Date"], errors="coerce")

# Convert numeric columns to numeric type
numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
for col in numeric_cols:
        combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce")
    
# Remove rows with missing Date or Close values/ 
combined_df = combined_df.dropna(subset=["Date", "Close"])

# Sort data by Stock and Date /
combined_df = combined_df.sort_values(by=["Stock", "Date"]).reset_index(drop=True)

#check missing values and preview cleaned data 
print (combined_df.isnull().sum())
print (combined_df.head())
print (combined_df.info())

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Stock     0
dtype: int64
        Date      Close       High        Low       Open   Volume Stock
0 2016-01-05  92.339996  92.849998  91.410004  92.010002  1821300  ADBE
1 2016-01-06  91.019997  91.500000  90.239998  91.099998  1674000  ADBE
2 2016-01-07  89.110001  91.400002  88.709999  89.400002  2717800  ADBE
3 2016-01-08  87.849998  89.989998  87.629997  89.680000  2263400  ADBE
4 2016-01-11  89.379997  89.800003  87.839996  88.070000  2362200  ADBE
<class 'pandas.DataFrame'>
RangeIndex: 25120 entries, 0 to 25119
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    25120 non-null  datetime64[us]
 1   Close   25120 non-null  float64       
 2   High    25120 non-null  float64       
 3   Low     25120 non-null  float64       
 4   Open    25120 non-null  float64       
 5   Volume  25120 non-null  int64         
 6   Stock   25120 non

 # Create return column

In [30]:
# Calculate daily percentgw return for each stock
combined_df["Return"] = combined_df.groupby("Stock")["Close"].pct_change()
#preview Date, stock, close and return columns 
print (combined_df[["Date", "Stock", "Close", "Return"]].head(10))

        Date Stock      Close    Return
0 2016-01-05  ADBE  92.339996       NaN
1 2016-01-06  ADBE  91.019997 -0.014295
2 2016-01-07  ADBE  89.110001 -0.020984
3 2016-01-08  ADBE  87.849998 -0.014140
4 2016-01-11  ADBE  89.379997  0.017416
5 2016-01-12  ADBE  89.820000  0.004923
6 2016-01-13  ADBE  88.300003 -0.016923
7 2016-01-14  ADBE  89.800003  0.016988
8 2016-01-15  ADBE  89.169998 -0.007016
9 2016-01-19  ADBE  89.180000  0.000112


# Initial eploratory data analysis